In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import requests
import zipfile
from pathlib import Path
import numpy as np
import warnings
import time
import os
import json

# ── Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, label_binarize
from sklearn.impute import SimpleImputer

# ── Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    BaggingClassifier, VotingClassifier, StackingClassifier
)
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV

# ── Evaluation
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_validate, RandomizedSearchCV
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, accuracy_score,
    roc_auc_score
)

# ── Optional libraries
try:
    from xgboost import XGBClassifier
    XGBOOST = True
    print("✓ XGBoost available")
except ImportError:
    XGBOOST = False
    print("✗ XGBoost not found (pip install xgboost)")

try:
    from lightgbm import LGBMClassifier
    LGBM = True
    print("✓ LightGBM available")
except ImportError:
    LGBM = False
    print("✗ LightGBM not found (pip install lightgbm)")

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_OK = True
    print("✓ SMOTE available")
except ImportError:
    SMOTE_OK = False
    print("✗ SMOTE not found (pip install imbalanced-learn)")

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_JOBS = -1
CV_FOLDS = 5
TUNE_ITER = 40

print("\nEnvironment ready.")

✓ XGBoost available
✓ LightGBM available
✓ SMOTE available

Environment ready.


 ## API Data Ingestion

In [2]:
import requests
import pandas as pd
import json
import time
import zipfile
from pathlib import Path
import numpy as np

# Local path where raw downloads are cached
RAW_DATA_DIR = Path("./data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Cache directory for fetched data (prevents re-fetching)
CACHE_DIR = Path("./data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Your API endpoints (now fixed)
API_BASE_URL = "https://melbourne-parking-api-nz2m.onrender.com"

# Optimal batch size (tested and working)
OPTIMAL_BATCH_SIZE = 5000  # 100x faster than before

# Melbourne Open Data API endpoints (for additional data like sign plates)
ON_STREET_SENSOR_API = "https://data.melbourne.vic.gov.au/api/records/1.0/search/"
SIGN_PLATES_API = "https://data.melbourne.vic.gov.au/api/records/1.0/search/"

# API dataset identifiers (for Melbourne Open Data)
ON_STREET_SENSOR_DATASET = "on-street-parking-bay-sensors"
SIGN_PLATES_DATASET = "sign-plates-located-in-each-parking-zone"

def warm_up_api(endpoint="sensors"):
    """Send a tiny request to wake up the API (handles cold start)."""
    try:
        url = f"{API_BASE_URL}/{endpoint}?limit=1&offset=0"
        print(f"Warming up API (handling cold start if needed)...")
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            print(f"API is awake and ready")
            return True
        else:
            print(f"API returned status {response.status_code}")
            return False
    except Exception as e:
        print(f"Warm-up ping failed: {e}")
        return False

def clean_id_columns(df, columns):
    """Remove commas from ID columns and convert to numeric."""
    for col in columns:
        if col in df.columns:
            # Convert to string, remove commas
            df[col] = df[col].astype(str).str.replace(",", "", regex=False)
            # Convert to numeric, coerce errors to NaN
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def fetch_sensors_data(max_records=100000, batch_size=OPTIMAL_BATCH_SIZE, use_cache=True, force_refresh=False):
    """
    Fetch sensors data - preserves original types, only cleans ID columns.
    """
    cache_file = CACHE_DIR / "sensors_full.parquet"
    
    # Check cache
    if use_cache and cache_file.exists() and not force_refresh:
        print(f"Loading sensors from cache...")
        df = pd.read_parquet(cache_file)
        print(f"Loaded {len(df):,} cached records")
        return df
    
    print(f"\nFetching sensors data with batch_size={batch_size}...")
    warm_up_api("sensors")
    
    all_data = []
    offset = 0
    
    while offset < max_records:
        url = f"{API_BASE_URL}/sensors?limit={batch_size}&offset={offset}"
        
        try:
            response = requests.get(url, timeout=60)
            
            if response.status_code == 200:
                data = response.json()
                batch = data.get("data", [])
                
                if not batch:
                    break
                
                # Keep original types - no string conversion
                all_data.extend(batch)
                offset += len(batch)
                print(f"  {len(all_data):,} / {max_records:,} records...")
                
                if len(batch) < batch_size:
                    break
                    
            else:
                print(f"  Stopped (status {response.status_code})")
                break
                
        except Exception as e:
            print(f"  Error: {e}")
            break
        
        time.sleep(0.05)
    
    if all_data:
        df = pd.DataFrame(all_data)
        
        # Clean ID columns (remove commas)
        id_columns = ["DeviceId", "StreetId", "DurationSeconds"]
        df = clean_id_columns(df, id_columns)
        
        # Convert boolean columns if present
        if "In Violation" in df.columns:
            df["In Violation"] = df["In Violation"].astype(bool)
        if "Vehicle Present" in df.columns:
            df["Vehicle Present"] = df["Vehicle Present"].astype(bool)
        
        # Convert Side Of Street to numeric
        if "Side Of Street" in df.columns:
            df["Side Of Street"] = pd.to_numeric(df["Side Of Street"], errors="coerce")
        
        # Save to cache
        if use_cache:
            df.to_parquet(cache_file, index=False)
            print(f"Cached {len(df):,} sensors records")
        
        print(f"Loaded {len(df):,} sensors records")
        print(f"Columns: {list(df.columns)}")
        return df
    
    return pd.DataFrame()

def fetch_events_data(max_records=100000, batch_size=OPTIMAL_BATCH_SIZE, use_cache=True, force_refresh=False):
    """
    Fetch events data - preserves original types, only cleans ID columns.
    """
    cache_file = CACHE_DIR / "events_full.parquet"
    
    # Check cache
    if use_cache and cache_file.exists() and not force_refresh:
        print(f"Loading events from cache...")
        df = pd.read_parquet(cache_file)
        print(f"Loaded {len(df):,} cached records")
        return df
    
    print(f"\nFetching events data with batch_size={batch_size}...")
    warm_up_api("events")
    
    all_data = []
    offset = 0
    
    while offset < max_records:
        url = f"{API_BASE_URL}/events?limit={batch_size}&offset={offset}"
        
        try:
            response = requests.get(url, timeout=60)
            
            if response.status_code == 200:
                data = response.json()
                batch = data.get("data", [])
                
                if not batch:
                    break
                
                # Keep original types - no string conversion
                all_data.extend(batch)
                offset += len(batch)
                print(f"  {len(all_data):,} / {max_records:,} records...")
                
                if len(batch) < batch_size:
                    break
                    
            else:
                print(f"  Stopped (status {response.status_code})")
                break
                
        except Exception as e:
            print(f"  Error: {e}")
            break
        
        time.sleep(0.05)
    
    if all_data:
        df = pd.DataFrame(all_data)
        
        # Clean ID columns (remove commas)
        id_columns = ["ParkingEventId", "DeviceId", "DurationSeconds", "StreetId", 
                      "BetweenStreet1 Id", "BetweenStreet2 Id", "BayID"]
        df = clean_id_columns(df, id_columns)
        
        # Convert boolean column
        if "InViolation" in df.columns:
            df["InViolation"] = df["InViolation"].astype(bool)
        
        # Convert numeric columns
        if "Area" in df.columns:
            df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
        if "SideOfStreet" in df.columns:
            df["SideOfStreet"] = pd.to_numeric(df["SideOfStreet"], errors="coerce")
        if "SignPlateId" in df.columns:
            df["SignPlateId"] = pd.to_numeric(df["SignPlateId"], errors="coerce")
        
        # Save to cache
        if use_cache:
            df.to_parquet(cache_file, index=False)
            print(f"Cached {len(df):,} events records")
        
        print(f"Loaded {len(df):,} events records")
        print(f"Columns: {list(df.columns)}")
        return df
    
    return pd.DataFrame()

def load_melbourne_api(dataset_slug, rows=5000):
    """Load a dataset from the Melbourne open data API and return a DataFrame."""
    url = "https://data.melbourne.vic.gov.au/api/records/1.0/search/"
    params = {"dataset": dataset_slug, "rows": rows}
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    records = data.get("records", [])
    return pd.DataFrame([record["fields"] for record in records])

def download_file(url, destination):
    """Download a file (streamed) and save to the destination path."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    with destination.open("wb") as output_file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                output_file.write(chunk)
    return destination

def read_cached_zip_csv(zip_url, cache_name, nrows=None):
    """Download (if needed) a zip file and return the first CSV inside as a DataFrame."""
    cache_path = RAW_DATA_DIR / cache_name
    if not cache_path.exists():
        download_file(zip_url, cache_path)
    with zipfile.ZipFile(cache_path) as archive:
        csv_member = next(
            (name for name in archive.namelist() if name.lower().endswith(".csv")),
            None,
        )
        if csv_member is None:
            raise ValueError(f"No CSV file found inside {cache_path.name}")
        with archive.open(csv_member) as csv_file:
            return pd.read_csv(csv_file, nrows=nrows)

## Load Datasets from APIs

In [4]:
print("Loading datasets from APIs...")
print("=" * 60)

# 1. Load events data
print("\n[1/4] Loading events data...")
try:
    parking_bay_arrivals_and_departures_df = fetch_events_data(max_records=100000, batch_size=5000)
    print(f"   Events: {len(parking_bay_arrivals_and_departures_df):,} records")
except Exception as e:
    print(f"   Events failed: {e}")
    parking_bay_arrivals_and_departures_df = pd.DataFrame()

# 2. Load sensors data
print("\n[2/4] Loading sensors data...")
try:
    on_street_car_parking_sensor_df = fetch_sensors_data(max_records=100000, batch_size=5000)
    print(f"   Sensors: {len(on_street_car_parking_sensor_df):,} records")
except Exception as e:
    print(f"   Sensors failed: {e}")
    on_street_car_parking_sensor_df = pd.DataFrame()

# 3. Load bay sensors from Melbourne Open Data
print("\n[3/4] Loading bay sensors from Melbourne Open Data...")
try:
    on_street_parking_bay_sensor_df = load_melbourne_api(ON_STREET_SENSOR_DATASET, rows=5000)
    print(f"   Bay sensors: {len(on_street_parking_bay_sensor_df):,} records")
except Exception as e:
    print(f"   Bay sensors failed: {e}")
    on_street_parking_bay_sensor_df = pd.DataFrame()

# 4. Load sign plates from Melbourne Open Data
print("\n[4/4] Loading sign plates data...")
try:
    sign_plates_located_in_parking_bays_df = load_melbourne_api(SIGN_PLATES_DATASET, rows=5000)
    print(f"   Sign plates: {len(sign_plates_located_in_parking_bays_df):,} records")
except Exception as e:
    print(f"   Sign plates failed: {e}")
    sign_plates_located_in_parking_bays_df = pd.DataFrame()

# Summary
print("\n" + "=" * 60)
print("DATA LOADING SUMMARY")
print("=" * 60)
print(f"  Events API:   {len(parking_bay_arrivals_and_departures_df):>10,} records")
print(f"  Sensors API:  {len(on_street_car_parking_sensor_df):>10,} records")
print(f"  Bay Sensors:  {len(on_street_parking_bay_sensor_df):>10,} records")
print(f"  Sign Plates:  {len(sign_plates_located_in_parking_bays_df):>10,} records")
print("=" * 60)
print("Data loading complete.")

Loading datasets from APIs...

[1/4] Loading events data...

Fetching events data with batch_size=5000...
Warming up API (handling cold start if needed)...
API is awake and ready
  5,000 / 100,000 records...
  10,000 / 100,000 records...
  15,000 / 100,000 records...
  20,000 / 100,000 records...
  25,000 / 100,000 records...
  30,000 / 100,000 records...
  35,000 / 100,000 records...
  40,000 / 100,000 records...
  45,000 / 100,000 records...
  50,000 / 100,000 records...
  55,000 / 100,000 records...
  60,000 / 100,000 records...
  65,000 / 100,000 records...
  70,000 / 100,000 records...
  75,000 / 100,000 records...
  80,000 / 100,000 records...
  85,000 / 100,000 records...
  90,000 / 100,000 records...
  95,000 / 100,000 records...
  100,000 / 100,000 records...
Cached 100,000 events records
Loaded 100,000 events records
Columns: ['ParkingEventId', 'DeviceId', 'ArrivalTime', 'DepartureTime', 'DurationSeconds', 'StreetMarker', 'SignPlateId', 'Sign', 'Area', 'AreaName', 'StreetId',

## Data Preview and Cleaning

In [5]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

if not on_street_car_parking_sensor_df.empty:
    on_street_car_parking_sensor_df = clean_column_names(on_street_car_parking_sensor_df)
if not on_street_parking_bay_sensor_df.empty:
    on_street_parking_bay_sensor_df = clean_column_names(on_street_parking_bay_sensor_df)
if not parking_bay_arrivals_and_departures_df.empty:
    parking_bay_arrivals_and_departures_df = clean_column_names(parking_bay_arrivals_and_departures_df)
if not sign_plates_located_in_parking_bays_df.empty:
    sign_plates_located_in_parking_bays_df = clean_column_names(sign_plates_located_in_parking_bays_df)

# Remove duplicates
if not on_street_car_parking_sensor_df.empty:
    on_street_car_parking_sensor_df = on_street_car_parking_sensor_df.drop_duplicates()
if not on_street_parking_bay_sensor_df.empty:
    on_street_parking_bay_sensor_df["location"] = (
        on_street_parking_bay_sensor_df["location"].apply(
            lambda value: tuple(value) if isinstance(value, list) else value
        )
    )
    on_street_parking_bay_sensor_df = on_street_parking_bay_sensor_df.drop_duplicates()
if not parking_bay_arrivals_and_departures_df.empty:
    parking_bay_arrivals_and_departures_df = parking_bay_arrivals_and_departures_df.drop_duplicates()
if not sign_plates_located_in_parking_bays_df.empty:
    sign_plates_located_in_parking_bays_df = sign_plates_located_in_parking_bays_df.drop_duplicates()

print("Data cleaning complete.")

Data cleaning complete.


 ## Data Processing and Merging

In [6]:
# Process numeric columns (remove commas, convert to numeric)
if not on_street_car_parking_sensor_df.empty:
    for col in ["deviceid", "streetid", "durationseconds"]:
        if col in on_street_car_parking_sensor_df.columns:
            on_street_car_parking_sensor_df[col] = (
                on_street_car_parking_sensor_df[col].astype(str).str.replace(",", "", regex=False)
            )
            on_street_car_parking_sensor_df[col] = pd.to_numeric(on_street_car_parking_sensor_df[col], errors="coerce")

if not parking_bay_arrivals_and_departures_df.empty:
    for col in ["parkingeventid", "deviceid", "durationseconds", "bayid"]:
        if col in parking_bay_arrivals_and_departures_df.columns:
            parking_bay_arrivals_and_departures_df[col] = (
                parking_bay_arrivals_and_departures_df[col].astype(str).str.replace(",", "", regex=False)
            )
            parking_bay_arrivals_and_departures_df[col] = pd.to_numeric(parking_bay_arrivals_and_departures_df[col], errors="coerce")

# Convert datetime columns
if not on_street_car_parking_sensor_df.empty:
    for col in ["arrivaltime", "departuretime"]:
        if col in on_street_car_parking_sensor_df.columns:
            on_street_car_parking_sensor_df[col] = pd.to_datetime(on_street_car_parking_sensor_df[col], errors="coerce")

if not on_street_parking_bay_sensor_df.empty:
    for col in ["lastupdated", "status_timestamp"]:
        if col in on_street_parking_bay_sensor_df.columns:
            on_street_parking_bay_sensor_df[col] = pd.to_datetime(on_street_parking_bay_sensor_df[col], errors="coerce", utc=True)

if not parking_bay_arrivals_and_departures_df.empty:
    for col in ["arrivaltime", "departuretime"]:
        if col in parking_bay_arrivals_and_departures_df.columns:
            parking_bay_arrivals_and_departures_df[col] = pd.to_datetime(parking_bay_arrivals_and_departures_df[col], errors="coerce")

# Remove timezone for easier handling
if not on_street_parking_bay_sensor_df.empty:
    for col in ["lastupdated", "status_timestamp"]:
        if col in on_street_parking_bay_sensor_df.columns:
            on_street_parking_bay_sensor_df[col] = on_street_parking_bay_sensor_df[col].dt.tz_convert(None)

print("Data conversion complete.")

Data conversion complete.


 ## Feature Engineering and Demand Aggregation

In [7]:
# Create occupied flag
if not on_street_parking_bay_sensor_df.empty and "status_description" in on_street_parking_bay_sensor_df.columns:
    on_street_parking_bay_sensor_df["occupied"] = on_street_parking_bay_sensor_df["status_description"].apply(
        lambda x: 1 if x == "Present" else 0
    )

# Rename parkingzone to zone_number for merging
if not sign_plates_located_in_parking_bays_df.empty and "parkingzone" in sign_plates_located_in_parking_bays_df.columns:
    sign_plates_located_in_parking_bays_df = sign_plates_located_in_parking_bays_df.rename(
        columns={"parkingzone": "zone_number"}
    )

# Merge bay sensor data with sign plates
merged_bay_df = None
if not on_street_parking_bay_sensor_df.empty and not sign_plates_located_in_parking_bays_df.empty:
    merged_bay_df = on_street_parking_bay_sensor_df.merge(
        sign_plates_located_in_parking_bays_df,
        on="zone_number",
        how="left"
    )
    # Drop rows without zone_number
    merged_bay_df = merged_bay_df.dropna(subset=["zone_number"])
    # Fill missing restriction details
    for col in ["restriction_days", "restriction_display", "time_restrictions_start", "time_restrictions_finish"]:
        if col in merged_bay_df.columns:
            merged_bay_df[col] = merged_bay_df[col].fillna("Unknown")
    
    # Check what columns are actually available
    print("Merged bay sensor data shape:", merged_bay_df.shape)
    print("Available columns:", list(merged_bay_df.columns))
    
    # Check if required columns exist, if not, create them
    if "status_day" not in merged_bay_df.columns:
        # Try alternative column names
        if "day" in merged_bay_df.columns:
            merged_bay_df["status_day"] = merged_bay_df["day"]
        elif "status_day" not in merged_bay_df.columns:
            print("'status_day' column not found. Creating from timestamp...")
            if "status_timestamp" in merged_bay_df.columns:
                merged_bay_df["status_day"] = pd.to_datetime(merged_bay_df["status_timestamp"]).dt.day_name()
            else:
                # Create synthetic day if needed
                merged_bay_df["status_day"] = "Monday"
    
    if "status_hour" not in merged_bay_df.columns:
        if "status_timestamp" in merged_bay_df.columns:
            merged_bay_df["status_hour"] = pd.to_datetime(merged_bay_df["status_timestamp"]).dt.hour
        else:
            merged_bay_df["status_hour"] = 12  # default
    
    if "is_weekend" not in merged_bay_df.columns:
        if "status_timestamp" in merged_bay_df.columns:
            merged_bay_df["is_weekend"] = pd.to_datetime(merged_bay_df["status_timestamp"]).dt.dayofweek >= 5
        else:
            merged_bay_df["is_weekend"] = False

# Aggregate demand by zone, day, hour, weekend
if merged_bay_df is not None and not merged_bay_df.empty:
    # Ensure we have all required columns
    required_cols = ["zone_number", "status_day", "status_hour", "is_weekend"]
    missing_cols = [col for col in required_cols if col not in merged_bay_df.columns]
    
    if missing_cols:
        print(f"Missing columns: {missing_cols}")
        print("Creating synthetic demand data as fallback...")
        np.random.seed(RANDOM_STATE)
        n_samples = 2000
        zones = merged_bay_df["zone_number"].unique() if "zone_number" in merged_bay_df.columns else np.random.choice(range(7000, 8000), n_samples)
        demand_df = pd.DataFrame({
            "zone_number": np.random.choice(zones, n_samples) if len(zones) > 0 else np.random.choice(range(7000, 8000), n_samples),
            "status_day": np.random.choice(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"], n_samples),
            "status_hour": np.random.randint(0, 24, n_samples),
            "is_weekend": np.random.choice([True, False], n_samples),
            "average_occupancy": np.random.uniform(0, 1, n_samples),
        })
        def demand_level(value):
            if value < 0.33:
                return "Low"
            elif value < 0.66:
                return "Medium"
            else:
                return "High"
        demand_df["demand_level"] = demand_df["average_occupancy"].apply(demand_level)
        print("Synthetic demand dataset shape:", demand_df.shape)
    else:
        demand_df = (
            merged_bay_df.groupby(
                ["zone_number", "status_day", "status_hour", "is_weekend"],
                as_index=False
            )["occupied"].mean()
        )
        demand_df = demand_df.rename(columns={"occupied": "average_occupancy"})

        def demand_level(value):
            if value < 0.33:
                return "Low"
            elif value < 0.66:
                return "Medium"
            else:
                return "High"

        demand_df["demand_level"] = demand_df["average_occupancy"].apply(demand_level)
        print("Demand dataset shape:", demand_df.shape)
else:
    # Create synthetic demand data if API calls failed
    print("Creating synthetic demand data for demonstration...")
    np.random.seed(RANDOM_STATE)
    n_samples = 2000
    zones = np.random.choice(range(7000, 8000), n_samples)
    days = np.random.choice(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"], n_samples)
    hours = np.random.randint(0, 24, n_samples)
    weekends = np.random.choice([True, False], n_samples)
    occupancies = np.random.uniform(0, 1, n_samples)
    
    demand_df = pd.DataFrame({
        "zone_number": zones,
        "status_day": days,
        "status_hour": hours,
        "is_weekend": weekends,
        "average_occupancy": occupancies,
    })
    
    def demand_level(value):
        if value < 0.33:
            return "Low"
        elif value < 0.66:
            return "Medium"
        else:
            return "High"
    
    demand_df["demand_level"] = demand_df["average_occupancy"].apply(demand_level)
    print("Synthetic demand dataset shape:", demand_df.shape)

print("\nTarget distribution:")
print(demand_df["demand_level"].value_counts())

Merged bay sensor data shape: (7766, 11)
Available columns: ['status_timestamp', 'zone_number', 'lastupdated', 'kerbsideid', 'status_description', 'location', 'occupied', 'restriction_display', 'time_restrictions_start', 'restriction_days', 'time_restrictions_finish']
'status_day' column not found. Creating from timestamp...
Demand dataset shape: (1657, 6)

Target distribution:
demand_level
Low       872
High      609
Medium    176
Name: count, dtype: int64


## Advanced Feature Engineering

In [8]:
df = demand_df.copy()

# Day ordinal mapping
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["status_day"] = pd.Categorical(df["status_day"], categories=day_order, ordered=True)
df["day_num"] = df["status_day"].cat.codes

# Cyclical encoding
df["hour_sin"] = np.sin(2 * np.pi * df["status_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["status_hour"] / 24)
df["day_sin"] = np.sin(2 * np.pi * df["day_num"] / 7)
df["day_cos"] = np.cos(2 * np.pi * df["day_num"] / 7)

# Time-of-day flags
df["is_peak_hour"] = df["status_hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)
df["is_business_hours"] = df["status_hour"].between(9, 17).astype(int)
df["is_night"] = df["status_hour"].isin(range(22, 24)).astype(int) | df["status_hour"].isin(range(0, 6)).astype(int)
if "is_weekend" in df.columns:
    df["is_weekend"] = df["is_weekend"].astype(int)

# Zone-level aggregate features
zone_hour = df.groupby(["zone_number", "status_hour"])["average_occupancy"].mean().rename("zone_hour_mean_occ")
zone_wknd = df.groupby(["zone_number", "is_weekend"])["average_occupancy"].mean().rename("zone_weekend_mean_occ")
df = df.join(zone_hour, on=["zone_number", "status_hour"])
df = df.join(zone_wknd, on=["zone_number", "is_weekend"])

# Encode target
LABEL_MAP = {"Low": 0, "Medium": 1, "High": 2}
df["target"] = df["demand_level"].map(LABEL_MAP)

print("Feature-engineered shape:", df.shape)
display(df.head(3))

Feature-engineered shape: (1657, 17)


,zone_number,status_day,status_hour,is_weekend,average_occupancy,demand_level,day_num,hour_sin,hour_cos,day_sin,day_cos,is_peak_hour,is_business_hours,is_night,zone_hour_mean_occ,zone_weekend_mean_occ,target
0,7010.0,Friday,1,0,0.0,Low,4,0.258819,0.965926,-0.433884,-0.900969,0,0,1,0.0,0.6,0
1,7010.0,Thursday,21,0,1.0,High,3,-0.707107,0.707107,0.433884,-0.900969,0,0,0,1.0,0.6,2
2,7010.0,Thursday,22,0,1.0,High,3,-0.500000,0.866025,0.433884,-0.900969,0,0,1,1.0,0.6,2


## Preprocessing Pipeline

In [9]:
NUMERIC_FEATS = [
    "status_hour",
    "hour_sin", "hour_cos",
    "day_sin", "day_cos",
]
BINARY_FEATS = [
    "is_weekend", "is_peak_hour", "is_business_hours", "is_night",
]
CAT_FEATS = ["status_day", "zone_number"]

ALL_FEATS = NUMERIC_FEATS + BINARY_FEATS + CAT_FEATS
TARGET = "target"

X = df[ALL_FEATS].copy()
y = df[TARGET].copy()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
binary_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATS),
    ("bin", binary_pipe, BINARY_FEATS),
    ("cat", cat_pipe, CAT_FEATS),
], remainder="drop")

print("Preprocessor ready.")
print(f"Total features entering models: {len(ALL_FEATS)}")

Preprocessor ready.
Total features entering models: 11


## Train / Test Split & Class Balancing

In [10]:
# Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Fit preprocessor ONLY on train
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# NO SMOTE - use class_weight='balanced' instead
print(f"Train: {X_train_proc.shape} | Test: {X_test_proc.shape}")
print("\nOriginal class distribution (train):")
print(pd.Series(y_train).value_counts().sort_index())
print("\nOriginal class distribution (test - untouched):")
print(pd.Series(y_test).value_counts().sort_index())

# Set class_weight parameter for models
CW = "balanced"  

Train: (1325, 11) | Test: (332, 11)

Original class distribution (train):
target
0    697
1    141
2    487
Name: count, dtype: int64

Original class distribution (test - untouched):
target
0    175
1     35
2    122
Name: count, dtype: int64


## SWEEP Test with using the real data division


In [11]:
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

SWEEP_MODELS = {
    "Dummy (baseline)": DummyClassifier(strategy="most_frequent"),

    # Fixed: Increased max_iter to 2000 to fix convergence warning
    "Logistic Regression": LogisticRegression(
        max_iter=2000,  # Changed from 1000
        class_weight=CW,
        random_state=RANDOM_STATE,
        solver="lbfgs"
    ),
    
    "Ridge Classifier": CalibratedClassifierCV(
        RidgeClassifier(class_weight=CW), cv=3
    ),

    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=15, n_jobs=1),  # Changed N_JOBS to 1

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, class_weight=CW, random_state=RANDOM_STATE
    ),

    "Bagging": BaggingClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=1  # Changed N_JOBS to 1
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight=CW,
        random_state=RANDOM_STATE, n_jobs=1  # Changed N_JOBS to 1
    ),
    
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200, class_weight=CW,
        random_state=RANDOM_STATE, n_jobs=1  # Changed N_JOBS to 1
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE
    ),

    "MLP (Neural Net)": MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=300,
        early_stopping=True, random_state=RANDOM_STATE
    ),

    "SVM (RBF)": SVC(
        kernel="rbf", class_weight=CW,
        probability=True, random_state=RANDOM_STATE
    ),
}

if XGBOOST:
    SWEEP_MODELS["XGBoost"] = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1  # Changed N_JOBS to 1
        # REMOVED: scale_pos_weight=CW (not used for multiclass)
    )

if LGBM:
    SWEEP_MODELS["LightGBM"] = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=1,  # Changed N_JOBS to 1
        class_weight=CW
    )

print(f"{len(SWEEP_MODELS)} models queued for sweep.")

14 models queued for sweep.


In [12]:
sweep_results = []

for name, model in SWEEP_MODELS.items():
    t0 = time.time()
    
    # Note: Using original X_train_proc (without SMOTE)
    cv_out = cross_validate(
        model, X_train_proc, y_train,  # Original train data, not SMOTE balanced
        cv=CV,
        scoring=["f1_weighted", "accuracy"],
        n_jobs=N_JOBS,
        return_train_score=False
    )
    elapsed = time.time() - t0

    row = {
        "model": name,
        "f1_mean": cv_out["test_f1_weighted"].mean(),
        "f1_std": cv_out["test_f1_weighted"].std(),
        "acc_mean": cv_out["test_accuracy"].mean(),
        "train_time_s": round(elapsed, 1),
    }
    sweep_results.append(row)
    print(f"{name:25s}  F1={row['f1_mean']:.4f} +/-{row['f1_std']:.4f}  ({elapsed:.1f}s)")

leaderboard = (
    pd.DataFrame(sweep_results)
    .sort_values("f1_mean", ascending=False)
    .reset_index(drop=True)
)
leaderboard.index += 1
print("\n=== LEADERBOARD ===")
display(leaderboard)

Dummy (baseline)           F1=0.3627 +/-0.0021  (2.0s)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Logistic Regression        F1=0.4042 +/-0.0376  (1.8s)
Ridge Classifier           F1=0.4714 +/-0.0167  (0.1s)
K-Nearest Neighbours       F1=0.5263 +/-0.0248  (0.1s)
Decision Tree              F1=0.4658 +/-0.0295  (0.0s)
Bagging                    F1=0.5430 +/-0.0192  (1.2s)
Random Forest              F1=0.5122 +/-0.0144  (1.3s)
Extra Trees                F1=0.4919 +/-0.0204  (1.4s)
AdaBoost                   F1=0.5321 +/-0.0276  (1.1s)
Gradient Boosting          F1=0.5529 +/-0.0090  (4.9s)
MLP (Neural Net)           F1=0.4863 +/-0.0313  (0.3s)
SVM (RBF)                  F1=0.3281 +/-0.0595  (0.9s)
XGBoost                    F1=0.5481 +/-0.0305  (1.6s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001089 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 329
[LightGBM] [Info] Number of data points in the train set: 1060, number of used features: 11
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001130 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 332
[LightGBM] [Info] Number of data points in the train set: 1060, number of used features: 11
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start trai

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,f1_mean,f1_std,acc_mean,train_time_s
1,LightGBM,0.555000,0.037330,0.551698,8.9
2,Gradient Boosting,0.552896,0.008987,0.566038,4.9
3,XGBoost,0.548076,0.030497,0.575094,1.6
4,Bagging,0.543040,0.019245,0.549434,1.2
5,AdaBoost,0.532142,0.027569,0.569811,1.1
6,K-Nearest Neighbours,0.526301,0.024827,0.571321,0.1
7,Random Forest,0.512190,0.014393,0.514717,1.3
8,Extra Trees,0.491879,0.020422,0.492075,1.4
9,MLP (Neural Net),0.486340,0.031326,0.541132,0.3
10,Ridge Classifier,0.471370,0.016683,0.541132,0.1


# SWEEP Test using SMOTE

In [13]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance the training classes
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)  # Using k_neighbors=3 because Medium class is small
X_train_smote, y_train_smote = smote.fit_resample(X_train_proc, y_train)

print(f"Original train shape: {X_train_proc.shape}")
print(f"SMOTE train shape: {X_train_smote.shape}")
print("\nClass distribution after SMOTE:")
print(pd.Series(y_train_smote).value_counts().sort_index())

Original train shape: (1325, 11)
SMOTE train shape: (2091, 11)

Class distribution after SMOTE:
target
0    697
1    697
2    697
Name: count, dtype: int64


In [14]:
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# With SMOTE, we DON'T need class_weight='balanced' anymore
SWEEP_MODELS_SMOTE = {
    "Dummy (baseline)": DummyClassifier(strategy="most_frequent"),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        solver="lbfgs"
    ),
    
    "Ridge Classifier": CalibratedClassifierCV(
        RidgeClassifier(), cv=3
    ),

    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=15, n_jobs=1),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, random_state=RANDOM_STATE
    ),

    "Bagging": BaggingClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=1
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE, n_jobs=1
    ),
    
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE, n_jobs=1
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE
    ),

    "MLP (Neural Net)": MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=300,
        early_stopping=True, random_state=RANDOM_STATE
    ),

    "SVM (RBF)": SVC(
        kernel="rbf",
        probability=True, random_state=RANDOM_STATE
    ),
}

if XGBOOST:
    SWEEP_MODELS_SMOTE["XGBoost"] = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1
    )

if LGBM:
    SWEEP_MODELS_SMOTE["LightGBM"] = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        n_jobs=1
    )

print(f"{len(SWEEP_MODELS_SMOTE)} models queued for sweep with SMOTE")

14 models queued for sweep with SMOTE


In [15]:
sweep_results_smote = []

for name, model in SWEEP_MODELS_SMOTE.items():
    t0 = time.time()
    
    cv_out = cross_validate(
        model, X_train_smote, y_train_smote,  # Using SMOTE balanced data
        cv=CV,
        scoring=["f1_weighted", "accuracy"],
        n_jobs=1,
        return_train_score=False,
        error_score='raise'
    )
    elapsed = time.time() - t0

    row = {
        "model": name,
        "f1_mean": cv_out["test_f1_weighted"].mean(),
        "f1_std": cv_out["test_f1_weighted"].std(),
        "acc_mean": cv_out["test_accuracy"].mean(),
        "train_time_s": round(elapsed, 1),
    }
    sweep_results_smote.append(row)
    print(f"{name:25s}  F1={row['f1_mean']:.4f} +/-{row['f1_std']:.4f}  ({elapsed:.1f}s)")

leaderboard_smote = (
    pd.DataFrame(sweep_results_smote)
    .sort_values("f1_mean", ascending=False)
    .reset_index(drop=True)
)
leaderboard_smote.index += 1
print("\n=== LEADERBOARD WITH SMOTE ===")
display(leaderboard_smote)

Dummy (baseline)           F1=0.1658 +/-0.0003  (0.0s)
Logistic Regression        F1=0.4112 +/-0.0291  (5.4s)
Ridge Classifier           F1=0.3954 +/-0.0248  (0.2s)
K-Nearest Neighbours       F1=0.5458 +/-0.0144  (0.1s)
Decision Tree              F1=0.5915 +/-0.0514  (0.1s)
Bagging                    F1=0.6696 +/-0.0135  (3.9s)
Random Forest              F1=0.6357 +/-0.0213  (3.6s)
Extra Trees                F1=0.6252 +/-0.0116  (2.7s)
AdaBoost                   F1=0.5047 +/-0.0278  (2.8s)
Gradient Boosting          F1=0.7059 +/-0.0160  (18.0s)
MLP (Neural Net)           F1=0.4016 +/-0.0301  (1.8s)
SVM (RBF)                  F1=0.3517 +/-0.0205  (3.6s)
XGBoost                    F1=0.6986 +/-0.0255  (3.0s)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000172 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Info] Number

,model,f1_mean,f1_std,acc_mean,train_time_s
1,LightGBM,0.713101,0.019986,0.714970,2.7
2,Gradient Boosting,0.705888,0.015988,0.707308,18.0
3,XGBoost,0.698570,0.025540,0.699652,3.0
4,Bagging,0.669577,0.013523,0.672401,3.9
5,Random Forest,0.635720,0.021287,0.639883,3.6
6,Extra Trees,0.625192,0.011649,0.629359,2.7
7,Decision Tree,0.591539,0.051442,0.592527,0.1
8,K-Nearest Neighbours,0.545751,0.014397,0.555228,0.1
9,AdaBoost,0.504746,0.027801,0.507407,2.8
10,Logistic Regression,0.411211,0.029061,0.417017,5.4


## SMOTE vs No SMOTE comparison

In [16]:
# Define label names for the classification report
label_names = ["Low", "Medium", "High"]

# Evaluate best model on test set
best_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=RANDOM_STATE
)
best_model.fit(X_train_smote, y_train_smote)

y_pred = best_model.predict(X_test_proc)

print(f"Test Set F1: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

Test Set F1: 0.5709

Classification Report:
              precision    recall  f1-score   support

         Low       0.63      0.70      0.66       175
      Medium       0.17      0.11      0.14        35
        High       0.58      0.55      0.56       122

    accuracy                           0.58       332
   macro avg       0.46      0.45      0.45       332
weighted avg       0.56      0.58      0.57       332



In [17]:
# Train No SMOTE version (using original training data)
best_model_no_smote = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=RANDOM_STATE
)
best_model_no_smote.fit(X_train_proc, y_train)
y_pred_no_smote = best_model_no_smote.predict(X_test_proc)

# Train SMOTE version (using SMOTE balanced data)
best_model_smote = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=RANDOM_STATE
)
best_model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = best_model_smote.predict(X_test_proc)

# Compare results
print("=" * 60)
print("TEST SET PERFORMANCE COMPARISON")
print("=" * 60)

print("\n--- NO SMOTE ---")
print(f"Weighted F1: {f1_score(y_test, y_pred_no_smote, average='weighted'):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_no_smote):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_no_smote, target_names=label_names))

print("\n--- WITH SMOTE ---")
print(f"Weighted F1: {f1_score(y_test, y_pred_smote, average='weighted'):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_smote):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote, target_names=label_names))

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"No SMOTE Test F1:  {f1_score(y_test, y_pred_no_smote, average='weighted'):.4f}")
print(f"SMOTE Test F1:     {f1_score(y_test, y_pred_smote, average='weighted'):.4f}")
print(f"Difference:        +{f1_score(y_test, y_pred_smote, average='weighted') - f1_score(y_test, y_pred_no_smote, average='weighted'):.4f}")

TEST SET PERFORMANCE COMPARISON

--- NO SMOTE ---
Weighted F1: 0.5739
Accuracy: 0.5843

Classification Report:
              precision    recall  f1-score   support

         Low       0.64      0.71      0.67       175
      Medium       0.35      0.20      0.25        35
        High       0.54      0.52      0.53       122

    accuracy                           0.58       332
   macro avg       0.51      0.47      0.48       332
weighted avg       0.57      0.58      0.57       332


--- WITH SMOTE ---
Weighted F1: 0.5709
Accuracy: 0.5813

Classification Report:
              precision    recall  f1-score   support

         Low       0.63      0.70      0.66       175
      Medium       0.17      0.11      0.14        35
        High       0.58      0.55      0.56       122

    accuracy                           0.58       332
   macro avg       0.46      0.45      0.45       332
weighted avg       0.56      0.58      0.57       332


SUMMARY
No SMOTE Test F1:  0.5739
SMOTE Test 

## Save Your Champion Model

In [21]:
import joblib

# Save the champion model
champion = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=RANDOM_STATE
)

# Train with sample weights for best performance
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
champion.fit(X_train_proc, y_train, sample_weight=sample_weights)

# Save model and preprocessor
joblib.dump(champion, "champion_parking_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

print("✅ Model saved successfully!")

✅ Model saved successfully!


## Create a Prediction Function

In [22]:
def predict_parking_demand(zone_number, day, hour, is_weekend):
    """
    Predict parking demand level.
    
    Parameters:
    zone_number: int (e.g., 7010)
    day: str (Monday, Tuesday, etc.)
    hour: int (0-23)
    is_weekend: bool (True/False)
    
    Returns:
    demand_level: str (Low/Medium/High)
    probabilities: dict with confidence scores
    """
    # Create input dataframe
    input_df = pd.DataFrame({
        "zone_number": [zone_number],
        "status_day": [day],
        "status_hour": [hour],
        "is_weekend": [is_weekend]
    })
    
    # Apply preprocessing
    input_processed = preprocessor.transform(input_df)
    
    # Predict
    prediction = champion.predict(input_processed)[0]
    probabilities = champion.predict_proba(input_processed)[0]
    
    # Map prediction to label
    label_map = {0: "Low", 1: "Medium", 2: "High"}
    
    return {
        "demand_level": label_map[prediction],
        "probabilities": {
            "Low": probabilities[0],
            "Medium": probabilities[1],
            "High": probabilities[2]
        }
    }

# Test the function
print("=" * 60)
print("TESTING PREDICTION FUNCTION")
print("=" * 60)

test_cases = [
    (7010, "Monday", 9, False),   # Monday 9 AM - Should be High
    (7010, "Sunday", 3, True),     # Sunday 3 AM - Should be Low
    (7010, "Thursday", 21, False), # Thursday 9 PM - Medium/High
]

for zone, day, hour, weekend in test_cases:
    result = predict_parking_demand(zone, day, hour, weekend)
    print(f"\nZone {zone}, {day}, {hour}:00, Weekend={weekend}")
    print(f"  → Demand: {result['demand_level']}")
    print(f"  → Confidence: L={result['probabilities']['Low']:.2f}, "
          f"M={result['probabilities']['Medium']:.2f}, "
          f"H={result['probabilities']['High']:.2f}")

TESTING PREDICTION FUNCTION


ValueError: columns are missing: {'is_peak_hour', 'is_night', 'day_sin', 'hour_cos', 'day_cos', 'hour_sin', 'is_business_hours'}

In [23]:
def predict_parking_demand(zone_number, day, hour, is_weekend):
    """
    Predict parking demand level.
    
    Parameters:
    zone_number: int (e.g., 7010)
    day: str (Monday, Tuesday, etc.)
    hour: int (0-23)
    is_weekend: bool (True/False)
    
    Returns:
    demand_level: str (Low/Medium/High)
    probabilities: dict with confidence scores
    """
    # Create input dataframe with ALL required features
    day_num = day_order.index(day) if day in day_order else 0
    
    input_df = pd.DataFrame({
        "zone_number": [zone_number],
        "status_day": [day],
        "status_hour": [hour],
        "is_weekend": [1 if is_weekend else 0],
        # Cyclical encoding
        "hour_sin": [np.sin(2 * np.pi * hour / 24)],
        "hour_cos": [np.cos(2 * np.pi * hour / 24)],
        "day_sin": [np.sin(2 * np.pi * day_num / 7)],
        "day_cos": [np.cos(2 * np.pi * day_num / 7)],
        # Time-of-day flags
        "is_peak_hour": [1 if hour in [7, 8, 9, 16, 17, 18] else 0],
        "is_business_hours": [1 if 9 <= hour <= 17 else 0],
        "is_night": [1 if hour >= 22 or hour <= 5 else 0],
    })
    
    # Apply preprocessing
    input_processed = preprocessor.transform(input_df)
    
    # Predict
    prediction = champion.predict(input_processed)[0]
    probabilities = champion.predict_proba(input_processed)[0]
    
    # Map prediction to label
    label_map = {0: "Low", 1: "Medium", 2: "High"}
    
    return {
        "demand_level": label_map[prediction],
        "probabilities": {
            "Low": probabilities[0],
            "Medium": probabilities[1],
            "High": probabilities[2]
        }
    }

# Test the function
print("=" * 60)
print("TESTING PREDICTION FUNCTION")
print("=" * 60)

test_cases = [
    (7010, "Monday", 9, False),   # Monday 9 AM - Should be High
    (7010, "Sunday", 3, True),     # Sunday 3 AM - Should be Low
    (7010, "Thursday", 21, False), # Thursday 9 PM - Medium/High
]

for zone, day, hour, weekend in test_cases:
    result = predict_parking_demand(zone, day, hour, weekend)
    print(f"\nZone {zone}, {day}, {hour}:00, Weekend={weekend}")
    print(f"  → Demand: {result['demand_level']}")
    print(f"  → Confidence: L={result['probabilities']['Low']:.2f}, "
          f"M={result['probabilities']['Medium']:.2f}, "
          f"H={result['probabilities']['High']:.2f}")

TESTING PREDICTION FUNCTION

Zone 7010, Monday, 9:00, Weekend=False
  → Demand: Low
  → Confidence: L=0.74, M=0.03, H=0.24

Zone 7010, Sunday, 3:00, Weekend=True
  → Demand: Low
  → Confidence: L=0.64, M=0.25, H=0.12

Zone 7010, Thursday, 21:00, Weekend=False
  → Demand: High
  → Confidence: L=0.11, M=0.01, H=0.88


## Document Your Results

In [25]:
# Create a summary report
print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

print("\n DATASET:")
print(f"  Total records: {len(X)}")
print(f"  Features: {X.shape[1]}")
print(f"  Zones: {df['zone_number'].nunique()}")

print("\n MODEL PERFORMANCE:")
print(f"  Champion: Gradient Boosting (No SMOTE)")
print(f"  Test F1 Score: 0.5739")
print(f"  Test Accuracy: 0.5843")

print("\n PER-CLASS PERFORMANCE:")
print(f"  Low Class F1:    0.67")
print(f"  Medium Class F1: 0.25")
print(f"  High Class F1:   0.53")

print("\n FILES SAVED:")
print("  - champion_parking_model.pkl")
print("  - preprocessor.pkl")

PROJECT SUMMARY

 DATASET:
  Total records: 1657
  Features: 11
  Zones: 332

 MODEL PERFORMANCE:
  Champion: Gradient Boosting (No SMOTE)
  Test F1 Score: 0.5739
  Test Accuracy: 0.5843

 PER-CLASS PERFORMANCE:
  Low Class F1:    0.67
  Medium Class F1: 0.25
  High Class F1:   0.53

 FILES SAVED:
  - champion_parking_model.pkl
  - preprocessor.pkl
